In [3]:
# Files
database_file = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/input_data/DIA/diat-barcode-taxa_harmonized.fasta"
primer_table = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/input_data/DIA/diat-barcode-primers.tsv"
run_name = 'DAI-TEST'
otl_folder = '/home/camilababo/Documents/DNAquaIMG/countries-otls/harmonized/dia'

In [4]:
# Intake data
from src.reference_database.sequence_import import *

custom_fasta_import = CustomFastaImport(database_file)
custom_fasta_import.read_fasta(database_file, check_taxid=False)
data = custom_fasta_import.data
data.head()

,seq_id,sequence,length,taxa_info
0,TCC6-18S-1,TAAGCCTGCATGTCTAAGTATAAACTGCTTATACGGTGAAACTGCG...,1688,Chlorogonium elongatum|Chlorogonium elongatum ...
1,TCC7a-18S-1,TAAGCCTGCATGTCTAAGTATAAATCTCTTACTTTGAAACTGCGAA...,1680,Fragilaria sp.|Fragilaria H.C.Lyngbye 1819|GEN...
2,TCC9-18S-1,GTATAAACTGCTATACTGTGAAACTGCGAATGGCTCATTAAATCAG...,1659,Pandorina morum|Pandorina morum O.F.Mull. Bory...
3,TCC26-18S-1,CTGCTTATACTGTGAAACTGCGAATGGCTCATTAAATCAGTTATAG...,1636,Scenedesmus spinosus|Scenedesmus spinosus Chod...
4,TCC27-18S-1,TAAACTGCTTATACTGTGAAACTGCGAATGGCTCATTAAATCAGTT...,1638,Scenedesmus quadricauda|Scenedesmus quadricaud...


In [5]:
processed_data = custom_fasta_import.pre_process_harmonized_fasta_database()
processed_data.head()

mozaiko INFO: This method assumes that the FASTA header has been harmonized andcontains the following pipe-separated information: 
sequence id, original taxonomy, harmonized taxonomy, rank, kingdom, phylum, class, order, family, genus.


,seq_id,sequence,length,all_taxa_info,original_taxa_info,scientificName,rank,kingdom,phylum,class,order,family,genus,species
0,TCC6-18S-1,TAAGCCTGCATGTCTAAGTATAAACTGCTTATACGGTGAAACTGCG...,1688,Chlorogonium elongatum|Chlorogonium elongatum ...,Chlorogonium elongatum,Chlorogonium elongatum P.A.Dang.,species,Plantae,Chlorophyta,Chlorophyceae,Volvocales,Haematococcaceae,Chlorogonium,Chlorogonium elongatum
1,TCC7a-18S-1,TAAGCCTGCATGTCTAAGTATAAATCTCTTACTTTGAAACTGCGAA...,1680,Fragilaria sp.|Fragilaria H.C.Lyngbye 1819|GEN...,Fragilaria sp.,Fragilaria H.C.Lyngbye 1819,genus,Chromista,Ochrophyta,Bacillariophyceae,Fragilariales,Fragilariaceae,Fragilaria,NaN
2,TCC9-18S-1,GTATAAACTGCTATACTGTGAAACTGCGAATGGCTCATTAAATCAG...,1659,Pandorina morum|Pandorina morum O.F.Mull. Bory...,Pandorina morum,Pandorina morum O.F.Mull. Bory,species,Plantae,Chlorophyta,Chlorophyceae,Volvocales,Volvocaceae,Pandorina,Pandorina morum
3,TCC26-18S-1,CTGCTTATACTGTGAAACTGCGAATGGCTCATTAAATCAGTTATAG...,1636,Scenedesmus spinosus|Scenedesmus spinosus Chod...,Scenedesmus spinosus,Scenedesmus spinosus Chodat,species,Plantae,Chlorophyta,Chlorophyceae,Sphaeropleales,Scenedesmaceae,Desmodesmus,Scenedesmus spinosus
4,TCC27-18S-1,TAAACTGCTTATACTGTGAAACTGCGAATGGCTCATTAAATCAGTT...,1638,Scenedesmus quadricauda|Scenedesmus quadricaud...,Scenedesmus quadricauda,Scenedesmus quadricauda Turpin Breb.,species,Plantae,Chlorophyta,Chlorophyceae,Sphaeropleales,Scenedesmaceae,Scenedesmus,Scenedesmus quadricauda


In [6]:
database_processed_path = custom_fasta_import.database_fasta_file

In [7]:
from src.in_silico_analysis.amplification import InSilicoAmplification

insil = InSilicoAmplification(database_processed_path, run_name=run_name)
insil.run_in_silico_analysis(primer_table)

mozaiko INFO: Checking if cutadapt is installed...
mozaiko INFO: cutadapt is installed.
mozaiko INFO: Checking if CRABS is installed...
mozaiko INFO: CRABS is installed.
mozaiko INFO: To continue the analysis, a set of primers is needed. This information should be uploaded as a TSV table and it should contain the following fields: ['target_group', 'barcode_region', 'assay_name', 'fwd_seq', 'rev_seq']
mozaiko INFO: All set. Running in-silico amplification...
mozaiko INFO: Running cutadapt command as: cutadapt -g ATGCGTTGGAGAGARCGTTTC;min_overlap=21...CAGTTGTWGGTAAATTAGAAGGTGATC;min_overlap=27 --output data/output_data/DAI-TEST/amplicon/rbcL_646F_998R.fasta /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/input_data/DIA/diat-barcode-taxa_harmonized_processed.fasta --no-indels -e 3 --revcomp --quiet --minimum-length 0 --maximum-length 552 --discard-untrimmed --action retain
mozaiko INFO: Running cutadapt command as: cutadapt -g ATGCGTTGGAGAGARCGTTTC;min_overlap=21...

In [8]:
from src.marker_scoring.metrics_system import *

output_folder = '/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/diat-barcode-test-fixed'

for otl in os.listdir(otl_folder):
    otl_path = os.path.join(otl_folder, otl)

    country_name = otl.split('_')[0]
    output_path = os.path.join(output_folder, country_name + '_ranked_primers.tsv')
    print(" ---------------------")
    print("Processing OTL:", otl)
    mex = MetricsSystemExecutor(results_folder=output_folder, otl=otl_path, primer_table=primer_table)
    ranked_df = mex.rank_primers(save_intermediate_ranks=True, output_path=output_path)
    mex.sort_otl_level_results(subdirectory_name=country_name)

 ---------------------
Processing OTL: Germany_taxonlist_diatoms_harmonized.tsv
Set insert_folder_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/diat-barcode-test-fixed/insert/filtered, amplicon_folder_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/diat-barcode-test-fixed/amplicon and incomplete_pbs_path to /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/diat-barcode-test-fixed/incomplete_pbs/filtered/filtered_intersection.
mozaiko INFO: Retrieving primer-PBS statistics.
mozaiko INFO: Retriving information on the Reference Database Quality.
mozaiko INFO: Calculating the number of barcodes per FASTA.
mozaiko INFO: Calculating the number of barcodes per FASTA.
results_folder: /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/mozaico/data/output_data/diat-barcode-test-fixed, insert_folder_path: None, amplicon_folder_path: None
Set insert_folder_path to /home/c

KeyboardInterrupt: 